# Bulk Confidence Evaluation

Dataset-level confidence metrics via `BulkStructuredModelEvaluator`. Accumulates (match, confidence) pairs across documents, computes metrics at overall and per-field levels.

In [ ]:
import random
from typing import Optional

from stickler.comparators import ExactComparator, LevenshteinComparator, NumericComparator
from stickler.structured_object_evaluator.models.comparable_field import ComparableField
from stickler.structured_object_evaluator.models.structured_model import StructuredModel
from stickler.structured_object_evaluator.bulk_structured_model_evaluator import BulkStructuredModelEvaluator
from stickler.structured_object_evaluator.models.confidence import AUROCMetric, BrierScoreMetric, ECEMetric, ErrorCaptureAtBudgetMetric

## 1. Model & Data Generator

In [ ]:
class Invoice(StructuredModel):
    invoice_number: str = ComparableField(comparator=ExactComparator(), threshold=1.0, weight=3.0)
    vendor: str = ComparableField(comparator=LevenshteinComparator(), threshold=0.7, weight=1.0)
    total: float = ComparableField(comparator=NumericComparator(tolerance=0.01), weight=2.0)
    notes: Optional[str] = ComparableField(comparator=LevenshteinComparator(), threshold=0.5, weight=0.5)

VENDORS = ["Acme Corp", "Globex Inc", "Initech LLC", "Umbrella Co", "Stark Industries"]
NOTES = ["Delivered to front door", "Left at reception", "Signed by manager", "Fragile items"]

def make_gt(i):
    return Invoice(
        invoice_number=f"INV-2024-{i:04d}",
        vendor=random.choice(VENDORS),
        total=round(random.uniform(100, 5000), 2),
        notes=random.choice(NOTES),
    )

def make_pred_calibrated(gt):
    """Realistic calibration: correct fields tend toward higher confidence,
    incorrect fields tend toward lower, but the ranges overlap."""
    d = {}
    for field, correct_rate, hi, lo in [
        ("invoice_number", 0.8, (0.55, 0.99), (0.10, 0.65)),
        ("vendor", 0.7, (0.50, 0.95), (0.15, 0.60)),
        ("total", 0.9, (0.55, 0.98), (0.10, 0.55)),
        ("notes", 0.6, (0.45, 0.90), (0.10, 0.55)),
    ]:
        val = getattr(gt, field)
        if random.random() < correct_rate:
            d[field] = {"value": val, "confidence": random.uniform(*hi)}
        else:
            wrong = f"WRONG-{random.randint(1,999)}" if isinstance(val, str) else val + random.uniform(50, 500)
            d[field] = {"value": wrong, "confidence": random.uniform(*lo)}
    return Invoice.from_json(d)

def make_pred_random_conf(gt):
    """Same accuracy, random confidence."""
    d = {}
    for field, correct_rate in [("invoice_number", 0.8), ("vendor", 0.7), ("total", 0.9), ("notes", 0.6)]:
        val = getattr(gt, field)
        if random.random() < correct_rate:
            d[field] = {"value": val, "confidence": random.uniform(0.1, 0.99)}
        else:
            wrong = f"WRONG-{random.randint(1,999)}" if isinstance(val, str) else val + random.uniform(50, 500)
            d[field] = {"value": wrong, "confidence": random.uniform(0.1, 0.99)}
    return Invoice.from_json(d)

random.seed(42)
ground_truths = [make_gt(i) for i in range(100)]
print(f"Generated {len(ground_truths)} ground truth documents")

## 2. Default Metrics (AUROC only)

In [ ]:
random.seed(42)
evaluator = BulkStructuredModelEvaluator(target_schema=Invoice)

for gt in ground_truths:
    evaluator.update(gt, make_pred_calibrated(gt))

results = evaluator.compute()

print(f"Documents: {results.document_count}")
print(f"\nconfidence_metrics structure:")
print(f"  keys: {list(results.confidence_metrics.keys())}")
print(f"  overall: {results.confidence_metrics['overall']}")
print(f"  fields:")
for field, m in results.confidence_metrics['fields'].items():
    print(f"    {field}: {m}")

## 3. Multiple Metrics (AUROC + Brier + ECE)

In [ ]:
random.seed(42)
evaluator_multi = BulkStructuredModelEvaluator(
    target_schema=Invoice,
    confidence_metrics=[AUROCMetric(), BrierScoreMetric(), ECEMetric(n_bins=10)]
)

for gt in ground_truths:
    evaluator_multi.update(gt, make_pred_calibrated(gt))

results_multi = evaluator_multi.compute()

print("Overall metrics:")
for name, val in results_multi.confidence_metrics['overall'].items():
    if name == 'ece':
        print(f"  {name}: value={val['value']:.4f}, bins={len(val['bins'])} bins")
    else:
        v = val['value']
        print(f"  {name}: {v:.4f}" if v is not None else f"  {name}: None")

## 4. ECE Bin Data (for reliability diagrams)

In [ ]:
ece_result = results_multi.confidence_metrics['overall']['ece']

print(f"ECE value: {ece_result['value']:.4f}")
print(f"\n{'Bin Range':<15} {'Count':<8} {'Accuracy':<12} {'Mean Conf':<12} {'Gap'}")
print("-" * 60)
for b in ece_result['bins']:
    lo, hi = b['range']
    gap = abs(b['accuracy'] - b['mean_confidence'])
    if b['count'] > 0:
        print(f"[{lo:.1f}, {hi:.1f})     {b['count']:<8} {b['accuracy']:<12.3f} {b['mean_confidence']:<12.3f} {gap:.3f}")
    else:
        print(f"[{lo:.1f}, {hi:.1f})     {b['count']:<8} {'--':<12} {'--':<12} --")

## 5. Error Capture at Review Budget

In [ ]:
random.seed(42)
eval_ecab = BulkStructuredModelEvaluator(
    target_schema=Invoice,
    confidence_metrics=[ErrorCaptureAtBudgetMetric(budgets=[0.10, 0.30, 0.50])]
)
for gt in ground_truths:
    eval_ecab.update(gt, make_pred_calibrated(gt))

results_ecab = eval_ecab.compute()
ecab = results_ecab.confidence_metrics['overall']['error_capture_at_budget']

print(f"Headline gain: {ecab['value']:.1f}x\n")
print(f"{'Budget':<12} {'Reviewed':<12} {'Errors Found':<15} {'Caught':<10} {'Gain'}")
print('-' * 60)
for budget, data in ecab['budgets'].items():
    print(f"{budget:<12.0%} {data['fields_reviewed']:<12} {data['errors_found']:<15} "
          f"{data['pct_errors_caught']:<10.0%} {data['gain']:.1f}x")

## 6. Well-Calibrated vs Random: Side-by-Side

In [ ]:
random.seed(42)
eval_good = BulkStructuredModelEvaluator(
    target_schema=Invoice,
    confidence_metrics=[AUROCMetric(), BrierScoreMetric()]
)
for gt in ground_truths:
    eval_good.update(gt, make_pred_calibrated(gt))

random.seed(42)
eval_bad = BulkStructuredModelEvaluator(
    target_schema=Invoice,
    confidence_metrics=[AUROCMetric(), BrierScoreMetric()]
)
for gt in ground_truths:
    eval_bad.update(gt, make_pred_random_conf(gt))

rg = eval_good.compute()
rb = eval_bad.compute()

print(f"{'Metric':<20} {'Calibrated':>12} {'Random Conf':>12}")
print("-" * 46)
for name in ['auroc', 'brier_score']:
    vg = rg.confidence_metrics['overall'][name]['value']
    vb = rb.confidence_metrics['overall'][name]['value']
    vg_s = f"{vg:.3f}" if vg is not None else "None"
    vb_s = f"{vb:.3f}" if vb is not None else "None"
    print(f"{name:<20} {vg_s:>12} {vb_s:>12}")

## 7. Per-Field Breakdown

In [ ]:
print(f"{'Field':<25} {'AUROC':>8} {'Brier':>8}")
print("-" * 43)
for field, metrics in rg.confidence_metrics['fields'].items():
    auroc = metrics['auroc']['value']
    brier = metrics['brier_score']['value']
    a_s = f"{auroc:.3f}" if auroc is not None else "None"
    b_s = f"{brier:.3f}" if brier is not None else "None"
    print(f"{field:<25} {a_s:>8} {b_s:>8}")

## 8. State Merge (Distributed Evaluation)

In [ ]:
random.seed(42)
wa = BulkStructuredModelEvaluator(target_schema=Invoice, confidence_metrics=[AUROCMetric()])
wb = BulkStructuredModelEvaluator(target_schema=Invoice, confidence_metrics=[AUROCMetric()])

for i, gt in enumerate(ground_truths):
    pred = make_pred_calibrated(gt)
    if i < 50:
        wa.update(gt, pred)
    else:
        wb.update(gt, pred)

wa.merge_state(wb.get_state())
merged = wa.compute()

# Compare with single-pass
random.seed(42)
single = BulkStructuredModelEvaluator(target_schema=Invoice, confidence_metrics=[AUROCMetric()])
for gt in ground_truths:
    single.update(gt, make_pred_calibrated(gt))
single_r = single.compute()

print(f"Merged AUROC:      {merged.confidence_metrics['overall']['auroc']['value']:.4f}")
print(f"Single-pass AUROC: {single_r.confidence_metrics['overall']['auroc']['value']:.4f}")
print(f"Match: {merged.confidence_metrics['overall']['auroc']['value'] == single_r.confidence_metrics['overall']['auroc']['value']}")

## 9. No Confidence Data

In [ ]:
eval_none = BulkStructuredModelEvaluator(target_schema=Invoice)
for gt in ground_truths[:10]:
    pred = Invoice(invoice_number=gt.invoice_number, vendor=gt.vendor, total=gt.total)
    eval_none.update(gt, pred)

r_none = eval_none.compute()
print(f"confidence_metrics: {r_none.confidence_metrics}")
print("No confidence data -> confidence_metrics is None")